# Saved NRMS / Gated NRMS: user–news latent space (inference only)

Based on `7_1_nrms_popularity_gated.ipynb`. Load a saved best seed checkpoint and the same prepared `nrms_features.pkl`. **No training, ablation or seed sweep.** Missing/incompatible checkpoints stop execution; no model is retrained.

Unlike concatenation-based MLP, this model scores news by a dot product with a user vector. For Gated NRMS, use the effective user vector `alpha*u_history + (1-alpha)*u_popularity`, exactly as the repository evaluator. Verify extracted dot products against the model's original forward call, including empty histories. Do not insert ReLU, project vectors further or normalise the vectors before reproducing scores.

Plot user/news vectors jointly with PCA, a single-impression score-coloured view, and clicked/unclicked histograms of dot product and cosine using blue palettes. Dot product is the actual scoring rule; cosine is a secondary diagnostic and omits vector norms. A dot-product objective does not guarantee Euclidean-distance clusters, perfect separation or better ranking than MLP.


## 1. Shared pipeline — refactor-final, outputs run_02


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/nguyenvanminh281005/bt4222-group12.git"
    BRANCH = "refactor-final"
    RUN_NAME = "run_02"
    BASE = Path("/content/drive/MyDrive/BT4222 - Group 12")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.metrics import ndcg_at_k, history_groups
import os
import re
import math
import time
import random
import json
from types import SimpleNamespace
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import rankdata
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configure style
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11
HIST_BINS = [-1, 0, 4, float("inf")]
HIST_LABELS = ["Cold Start (0)", "Sparse (1-4)", "Warm (>=5)"]
_TIE_RNG = np.random.default_rng(0)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from sklearn.decomposition import PCA
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
import hashlib

REFERENCE_RUN_NAME="run_01"
MODEL_TAG="gated"  # use "nrms" to inspect the saved plain NRMS checkpoint
SEED=42
SPLIT="test"  # "val" is also supported
SAMPLE_IMPRESSIONS=500
ENCODE_BATCH=512
USER_BATCH=128
VERIFY_FORWARD_RECORDS=12  # full slates, stratified across available history cohorts
SOURCE_N_HEADS=16  # matches 7_1; head count cannot be inferred from weight shapes
REFERENCE_NRMS_DIR=Path(PATHS["outputs"]).parent/REFERENCE_RUN_NAME/"nrms"
CHECKPOINT=REFERENCE_NRMS_DIR/"checkpoints"/f"{MODEL_TAG}_best_seed{SEED}.pt"
SUMMARY_PATH=REFERENCE_NRMS_DIR/"figures"/("results_summary.json" if MODEL_TAG=="nrms" else f"results_summary_{MODEL_TAG}.json")
OUT_DIR=Path(PATHS["outputs"])/"nrms"/f"latent_{MODEL_TAG}_seed{SEED}"
OUT_DIR.mkdir(parents=True,exist_ok=True)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
BLUE_LIGHT,BLUE_MEDIUM,BLUE_DARK="#9ecae1","#4292c6","#08519c"
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR/name,dpi=180,bbox_inches="tight")
    display(plt.gcf())
    plt.close()
print("Read-only checkpoint:",CHECKPOINT)
print("New plots and tables:",OUT_DIR)


## 2. Exact source architectures — no training code


In [ ]:
class AttentionPooling(nn.Module):
    """
    Additive Attention (Attention Pooling) layer as formulated in NRMS (Wu et al., 2019).
    Computes energy a_i = q^T tanh(W * h_i + b), followed by softmax normalization.
    """

    def __init__(self, d_model=256, query_dim=200):
        super().__init__()
        self.proj = nn.Linear(d_model, query_dim)
        self.query = nn.Parameter(torch.empty(query_dim).uniform_(-0.1, 0.1))

    def forward(self, x, mask=None):
        # x: [B, L, D]
        energy = torch.tanh(self.proj(x))  # [B, L, query_dim]
        scores = torch.matmul(energy, self.query)  # [B, L]
        if mask is not None:
            all_pad = (~mask).all(dim=-1, keepdim=True)
            scores = scores.masked_fill(~mask, -1e9)
        alpha = F.softmax(scores, dim=-1)  # [B, L]
        if mask is not None:
            alpha = alpha.masked_fill(all_pad, 0.0)
        out = torch.sum(x * alpha.unsqueeze(-1), dim=1)  # [B, D]
        return out, alpha


In [ ]:
class NewsEncoder(nn.Module):
    """
    News Encoder:
    Word Embedding (GloVe 300d) -> Projection (256d) -> Multi-Head Self-Attention (16 heads) -> Additive Attention (200d query)
    """

    def __init__(
        self,
        pretrained_embed,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_embed, freeze=False, padding_idx=0
        )
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(pretrained_embed.shape[1], d_model)
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, tokens, mask=None, return_weights=False):
        # tokens: [N, L]
        if mask is None:
            mask = tokens > 0
        x = self.embedding(tokens)
        x = self.dropout(x)
        x = self.proj(x)
        if self.use_self_attn:
            key_padding_mask = ~mask
            all_pad = key_padding_mask.all(dim=-1)
            if all_pad.any():
                key_padding_mask = key_padding_mask.clone()
                key_padding_mask[all_pad] = False
            attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
            attn_out = self.dropout(attn_out)
        else:
            attn_out = x
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


In [ ]:
class UserEncoder(nn.Module):
    """
    User Encoder:
    News-level Multi-Head Self-Attention (16 heads) -> Additive Attention (200d query)
    """

    def __init__(
        self, d_model=256, n_heads=16, query_dim=200, dropout=0.2, use_self_attn=True
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, news_vecs, mask=None, return_weights=False):
        # news_vecs: [B, P, D]
        if self.use_self_attn:
            if mask is not None:
                key_padding_mask = ~mask
                all_pad = key_padding_mask.all(dim=-1)
                if all_pad.any():
                    key_padding_mask = key_padding_mask.clone()
                    key_padding_mask[all_pad] = False
            else:
                key_padding_mask = None
            attn_out, _ = self.self_attn(
                news_vecs, news_vecs, news_vecs, key_padding_mask=key_padding_mask
            )
            attn_out = self.dropout(attn_out)
        else:
            attn_out = news_vecs
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


In [ ]:
class NRMS(nn.Module):
    """
    Complete NRMS Architecture:
    Shared News Encoder + User Encoder + Dot Product Click Predictor

    pop_sets (optional): [n_days, K] news indices of each day's popular set (0 = pad).
    Only used by the hard cold-start fallback at evaluation time.
    """

    def __init__(
        self,
        pretrained_embed,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
        pop_sets=None,
    ):
        super().__init__()
        self.register_buffer("news_title_lookup", news_title_lookup)
        self.register_buffer(
            "pop_sets",
            pop_sets if pop_sets is not None else torch.zeros(0, 0, dtype=torch.long),
            persistent=False,
        )
        self.news_encoder = NewsEncoder(
            pretrained_embed,
            d_model,
            n_heads,
            query_dim,
            dropout,
            use_self_attn=use_self_attn,
        )
        self.user_encoder = UserEncoder(
            d_model, n_heads, query_dim, dropout, use_self_attn=use_self_attn
        )

    def forward(self, hist_ids, cand_ids, day_idx=None):
        # hist_ids: [B, P], cand_ids: [B, C]
        B, P = hist_ids.shape
        C = cand_ids.shape[1]
        all_ids = torch.cat([hist_ids.view(-1), cand_ids.view(-1)])
        unique_ids, inv = torch.unique(all_ids, return_inverse=True)
        unique_tokens = self.news_title_lookup[unique_ids]
        unique_vecs = self.news_encoder(unique_tokens)
        all_vecs = unique_vecs[inv]
        hist_news_vecs = all_vecs[: B * P].view(B, P, -1)
        cand_news_vecs = all_vecs[B * P :].view(B, C, -1)
        hist_mask = hist_ids > 0
        user_vecs = self.user_encoder(hist_news_vecs, mask=hist_mask)
        scores = torch.bmm(cand_news_vecs, user_vecs.unsqueeze(2)).squeeze(2)
        return scores


In [ ]:
class GatedNRMS(NRMS):
    """
    Two-branch NRMS with a learned per-user gate:
      - History branch:    u_hist = UserEncoder(history), or UserEncoder([EMPTY]) if the history is empty
      - Popularity branch: u_pop  = UserEncoder(previous day's category-stratified popular articles)
      - Gate:              alpha  = sigmoid(MLP([u_hist ; u_pop ; log(1+|H|) / log(1+P)]))
      - Score:             s(c)   = alpha * <u_hist, e_c> + (1 - alpha) * <u_pop, e_c>
    No hand-set threshold: the [EMPTY] vector lets the gate learn when to trust the popularity branch.
    """

    def __init__(
        self,
        pretrained_embed,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
        pop_sets=None,
        gate_hidden=128,
        max_hist_len=50,
    ):
        super().__init__(
            pretrained_embed,
            news_title_lookup,
            d_model,
            n_heads,
            query_dim,
            dropout,
            use_self_attn=use_self_attn,
            pop_sets=pop_sets,
        )
        self.max_hist_len = max_hist_len
        self.empty_vec = nn.Parameter(torch.empty(d_model).normal_(0.0, 0.1))
        self.gate = nn.Sequential(
            nn.Linear(2 * d_model + 1, gate_hidden),
            nn.ReLU(),
            nn.Linear(gate_hidden, 1),
        )

    def encode_popular(self, pop_news_vecs):
        # pop_news_vecs: [n_days, K, D] -> u_pop per day: [n_days, D]
        return self.user_encoder(pop_news_vecs, mask=(self.pop_sets > 0))

    def gate_users(self, hist_news_vecs, hist_mask, u_pop):
        # hist_news_vecs: [B, P, D], hist_mask: [B, P], u_pop: [B, D]
        B, P, D = hist_news_vecs.shape
        n_hist = hist_mask.sum(dim=1)
        empty = n_hist == 0
        # Empty history -> a single learnable [EMPTY] item in the last slot
        slot = torch.zeros_like(hist_mask)
        slot[:, -1] = True
        fill = slot & empty.unsqueeze(1)
        hist_news_vecs = torch.where(
            fill.unsqueeze(-1), self.empty_vec.expand(B, P, D), hist_news_vecs
        )
        hist_mask = hist_mask | fill
        u_hist = self.user_encoder(hist_news_vecs, mask=hist_mask)
        len_feat = (
            torch.log1p(n_hist.float()) / math.log1p(self.max_hist_len)
        ).unsqueeze(1)
        alpha = torch.sigmoid(
            self.gate(torch.cat([u_hist, u_pop, len_feat], dim=1))
        ).squeeze(1)
        return u_hist, alpha

    def forward(self, hist_ids, cand_ids, day_idx=None, return_alpha=False):
        B, P = hist_ids.shape
        C = cand_ids.shape[1]
        n_days, K = self.pop_sets.shape
        all_ids = torch.cat(
            [hist_ids.view(-1), cand_ids.view(-1), self.pop_sets.view(-1)]
        )
        unique_ids, inv = torch.unique(all_ids, return_inverse=True)
        all_vecs = self.news_encoder(self.news_title_lookup[unique_ids])[inv]
        hist_news_vecs = all_vecs[: B * P].view(B, P, -1)
        cand_news_vecs = all_vecs[B * P : B * P + B * C].view(B, C, -1)
        pop_news_vecs = all_vecs[B * P + B * C :].view(n_days, K, -1)
        u_pop = self.encode_popular(pop_news_vecs)[day_idx]  # [B, D]
        u_hist, alpha = self.gate_users(hist_news_vecs, hist_ids > 0, u_pop)
        s_hist = torch.bmm(cand_news_vecs, u_hist.unsqueeze(2)).squeeze(2)
        s_pop = torch.bmm(cand_news_vecs, u_pop.unsqueeze(2)).squeeze(2)
        scores = alpha.unsqueeze(1) * s_hist + (1 - alpha.unsqueeze(1)) * s_pop
        if return_alpha:
            return scores, alpha
        return scores


In [ ]:
def eval_impression(labels, scores):
    """
    Evaluates AUC, MRR, NDCG@5, and NDCG@10 for a single impression slate.
    Ties are not resolved by the logged slate order (a position artifact): AUC uses average ranks,
    so fully tied scores give exactly 0.5, and MRR/NDCG break ties uniformly at random.
    """
    labels = np.asarray(labels, dtype=np.int32)
    scores = np.asarray(scores, dtype=np.float32)
    pos_mask = labels == 1
    n_pos = pos_mask.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        auc = 0.5
    else:
        ranks = rankdata(scores)
        auc = (ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    perm = _TIE_RNG.permutation(len(scores))
    desc_order = perm[np.argsort(-scores[perm], kind="stable")]
    sorted_labels = labels[desc_order]
    pos_ranks = np.where(sorted_labels == 1)[0]
    mrr = 1.0 / (pos_ranks[0] + 1) if len(pos_ranks) > 0 else 0.0
    return (
        float(auc),
        float(mrr),
        ndcg_at_k(labels, sorted_labels, 5),
        ndcg_at_k(labels, sorted_labels, 10),
    )


## 3. Restore best weights with the original news-index mapping
The training notebook saves `{tag}_best_seed42.pt` as a plain state dict when validation nDCG@5 improves. Defaults here read `run_01/nrms/checkpoints/gated_best_seed42.pt`; new diagnostics go to run_02. Change only the reference settings if the existing checkpoint lives elsewhere.

Verify the saved `news_title_lookup` exactly matches the prepared bundle. Reconstruct dimensions from checkpoint tensors; use the source architecture's 16 heads and history length 50. Popular sets are non-persistent buffers, so they must come from the original feature bundle. Check the saved summary's popularity configuration when available; the original training format does not store a full feature-bundle hash in its state dict. A matching lookup is a mapping check, not a cryptographic verification of every historical/popularity input.


In [ ]:
def load_saved_nrms(checkpoint,payload,n_heads=SOURCE_N_HEADS):
    checkpoint=Path(checkpoint)
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Saved NRMS checkpoint missing: {checkpoint}. Set CHECKPOINT to the existing file; training is disabled.")
    state=torch.load(checkpoint,map_location="cpu",weights_only=True)
    lookup=torch.as_tensor(payload["news_title_lookup"],dtype=torch.long,device="cpu")
    if "news_title_lookup" not in state or not torch.equal(state["news_title_lookup"].cpu(),lookup):
        raise ValueError("Checkpoint news_title_lookup differs from the prepared bundle. Use the original news-index/vocabulary mapping.")
    gated="empty_vec" in state
    d_model=state["news_encoder.proj.weight"].shape[0]
    query_dim=state["news_encoder.pooling.proj.weight"].shape[0]
    self_attention="news_encoder.self_attn.in_proj_weight" in state
    if self_attention and d_model%n_heads:
        raise ValueError("Attention head count is incompatible with checkpoint model dimension")
    max_hist=int(payload["max_hist_len"])
    if max_hist!=50:
        raise ValueError("The source 7_1 model/evaluator uses history length 50; select its original feature bundle")
    pop=torch.as_tensor(payload["pop_sets"],dtype=torch.long,device="cpu")
    if gated and (pop.ndim!=2 or pop.numel()==0):
        raise ValueError("Gated checkpoint requires the original nonempty daily popularity sets")
    kwargs=dict(d_model=d_model,n_heads=n_heads,query_dim=query_dim,dropout=.2,
        use_self_attn=self_attention,pop_sets=pop)
    if gated:
        kwargs.update(gate_hidden=state["gate.0.weight"].shape[0],max_hist_len=50)
    cls=GatedNRMS if gated else NRMS
    model=cls(torch.zeros_like(state["news_encoder.embedding.weight"]),lookup,**kwargs)
    model.load_state_dict(state,strict=True)
    return model.to(device).eval(),gated,d_model

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing {CHECKPOINT}; this notebook never trains a replacement.")
payload=pd.read_pickle(PATHS["features"]/"nrms_features.pkl")
if SUMMARY_PATH.is_file():
    saved_summary=json.loads(SUMMARY_PATH.read_text())
    cfg=saved_summary["config"]
    expected_pop=dict(per_cat=cfg["pop_per_cat"],extra=cfg["pop_extra"],min_cat_clicks=cfg["pop_min_cat_clicks"])
else:
    expected_pop=dict(per_cat=2,extra=10,min_cat_clicks=20)
    print("Saved summary unavailable; checking the source notebook's default popularity configuration.")
if payload["popularity_config"]!=expected_pop:
    raise ValueError("Prepared popularity configuration does not match the reference training configuration")
model,IS_GATED,D_MODEL=load_saved_nrms(CHECKPOINT,payload)
print("Loaded:","GatedNRMS" if IS_GATED else "NRMS","dimension:",D_MODEL,"heads:",SOURCE_N_HEADS)


## 4. Sample impressions and extract the actual scoring vectors
Uniformly sample impressions, retaining all their candidates and observed labels. User vectors belong to **user-impressions**, not a permanent user ID: histories and daily popularity can change. Encode only news IDs required by the sampled histories/candidates and daily popularity sets, once per checkpoint. No feature engineering or model fitting is performed.

Plain NRMS uses the UserEncoder's history vector, with no hard popularity fallback. Empty history produces a zero user vector and tied zero scores; cosine is undefined for zero vectors and is recorded as missing. Gated NRMS uses its original learned EMPTY representation, gate and popular branch. Dot products retain vector norms; L2-normalised cosine does not reproduce the model score.


In [ ]:
records=payload[f"{SPLIT}_records"]
if not records:raise ValueError("No evaluation impressions in selected split")
rng=np.random.default_rng(SEED)
sample_ids=np.sort(rng.choice(len(records),min(SAMPLE_IMPRESSIONS,len(records)),replace=False))
sample_records=[records[int(i)] for i in sample_ids]

@torch.no_grad()
def extract_latent(model,records):
    gated=isinstance(model,GatedNRMS)
    model.eval()
    needed={0}
    for rec in records:
        if len(rec["history"])!=50 or len(rec["candidates"])!=len(rec["labels"]):
            raise ValueError("Evaluation record differs from the source history/candidate contract")
        needed.update(rec["history"])
        needed.update(rec["candidates"])
    if gated:needed.update(model.pop_sets.flatten().cpu().tolist())
    news_ids=np.array(sorted(needed),dtype=np.int64)
    if news_ids[0]<0 or news_ids[-1]>=len(model.news_title_lookup):
        raise ValueError("News IDs fall outside the checkpoint lookup table")
    dim=model.news_encoder.proj.out_features
    vectors=torch.zeros(len(model.news_title_lookup),dim,device=device)
    for start in range(0,len(news_ids),ENCODE_BATCH):
        ids=torch.as_tensor(news_ids[start:start+ENCODE_BATCH],device=device)
        vectors[ids]=model.news_encoder(model.news_title_lookup[ids])
    vectors[0]=0
    if gated:
        popular=model.encode_popular(vectors[model.pop_sets])
    users,alphas,history_vectors,popular_vectors=[],[],[],[]
    candidate_ids,pair_group,labels,raw_scores=[],[],[],[]
    verify=set()
    for predicate in (lambda n:n==0,lambda n:0<n<5,lambda n:n>=5):
        match=next((i for i,r in enumerate(records) if predicate(r["history_len"])),None)
        if match is not None:verify.add(match)
    verify.update(range(min(max(VERIFY_FORWARD_RECORDS,3),len(records))))
    verified_error=0.0
    for start in range(0,len(records),USER_BATCH):
        chunk=records[start:start+USER_BATCH]
        hist=torch.tensor([r["history"] for r in chunk],dtype=torch.long,device=device)
        day=torch.tensor([r.get("day_idx",0) for r in chunk],dtype=torch.long,device=device)
        mask=hist>0
        if gated:
            if torch.any(day<0) or torch.any(day>=len(popular)):
                raise ValueError("Impression day index does not match saved popular-set inputs")
            upop=popular[day]
            uhist,alpha=model.gate_users(vectors[hist],mask,upop)
            user=alpha[:,None]*uhist+(1-alpha[:,None])*upop
        else:
            uhist=model.user_encoder(vectors[hist],mask=mask)
            user=uhist
            alpha=torch.ones(len(chunk),device=device)
            upop=torch.zeros_like(user)
        users.append(user.cpu().numpy())
        history_vectors.append(uhist.cpu().numpy())
        popular_vectors.append(upop.cpu().numpy())
        alphas.append(alpha.cpu().numpy())
        for j,rec in enumerate(chunk):
            ids=torch.as_tensor(rec["candidates"],dtype=torch.long,device=device)
            dot=vectors[ids]@user[j]
            # Verify complete slates for a small cohort-covering subset; avoid re-encoding
            # every sampled article hundreds of times just to repeat this check.
            if start+j in verify:
                actual=model(hist[j:j+1],ids[None,:],day[j:j+1])[0]
                if not torch.allclose(dot,actual,atol=1e-4,rtol=1e-4):
                    raise AssertionError("Extracted user/news dot products differ from original forward scores")
                verified_error=max(verified_error,float((dot-actual).abs().max().item()))
            candidate_ids.extend(rec["candidates"])
            pair_group.extend([start+j]*len(ids))
            labels.extend(rec["labels"])
            raw_scores.extend(dot.cpu().tolist())
    return dict(news_ids=news_ids,news_vectors=vectors[news_ids].cpu().numpy(),
        user_vectors=np.concatenate(users),history_vectors=np.concatenate(history_vectors),
        popularity_vectors=np.concatenate(popular_vectors),alpha=np.concatenate(alphas),
        candidate_ids=np.array(candidate_ids,dtype=np.int64),pair_group=np.array(pair_group,dtype=np.int64),
        labels=np.array(labels,dtype=np.int64),scores=np.array(raw_scores),verified_max_error=verified_error,verified_slates=len(verify))

latent=extract_latent(model,sample_records)
np.savez_compressed(OUT_DIR/"nrms_latent_vectors.npz",**latent,sampled_record_indices=sample_ids)
print("Verified original forward vs extracted dot product, max absolute error:",latent["verified_max_error"])
print("Sample impressions:",len(sample_records),"candidate exposures:",len(latent["labels"]))


## 5. Joint PCA, blue score/label views and distribution histograms
Fit **one PCA** on the raw user-impression vectors plus unique sampled candidate news vectors. Transform both sides using that same fitted PCA. Do not fit separate user/news projections and compare their coordinates. Retain norms because the model uses raw dot product.

Left: all sampled user vectors and news exposures coloured clicked/unclicked (one article can appear in both label groups across impressions). Right: one reproducibly chosen warm impression with both labels; candidates coloured by its actual dot-product scores, with the same user/news PCA coordinates. A single impression avoids assigning different user-dependent scores to one global article point. A source user vector may still look far away in PCA: dot product is not Euclidean distance and PCA loses dimensions.

Bottom: density-normalised raw dot-product and cosine histograms, each label separately normalised. Histogram overlap is descriptive, bin-dependent and does not equal error rate. Raw scores need not be calibrated across impressions; cosine ignores magnitude and is missing for zero-norm users/news. Neither plot replaces within-impression nDCG.


In [ ]:
news_map={int(n):i for i,n in enumerate(latent["news_ids"])}
candidate_rows=np.array([news_map[int(i)] for i in latent["candidate_ids"]])
candidate_vectors=latent["news_vectors"][candidate_rows]
pair_users=latent["user_vectors"][latent["pair_group"]]
user_norm=np.linalg.norm(pair_users,axis=1)
news_norm=np.linalg.norm(candidate_vectors,axis=1)
denominator=user_norm*news_norm
cosine=np.divide(latent["scores"],denominator,out=np.full(len(denominator),np.nan),where=denominator>1e-12)
unique_candidate_ids=np.unique(latent["candidate_ids"])
unique_candidate_rows=np.array([news_map[int(i)] for i in unique_candidate_ids])
pca=PCA(n_components=2)
pca.fit(np.vstack([latent["user_vectors"],latent["news_vectors"][unique_candidate_rows]]))
user_xy=pca.transform(latent["user_vectors"])
news_xy=pca.transform(candidate_vectors)
np.savez(OUT_DIR/"nrms_joint_pca.npz",user_xy=user_xy,news_xy=news_xy,
    components=pca.components_,center=pca.mean_,explained_variance_ratio=pca.explained_variance_ratio_)
fig,axes=plt.subplots(2,2,figsize=(13,10))
for label,color in ((0,BLUE_LIGHT),(1,BLUE_DARK)):
    mask=latent["labels"]==label
    axes[0,0].scatter(news_xy[mask,0],news_xy[mask,1],color=color,s=8,alpha=.3,
        label="Clicked news exposure" if label else "Unclicked news exposure")
axes[0,0].scatter(user_xy[:,0],user_xy[:,1],color=BLUE_MEDIUM,marker="x",s=20,alpha=.6,label="User-impression")
axes[0,0].set(title=f"Joint user–news PCA (retains {pca.explained_variance_ratio_.sum():.1%})",xlabel="PC1",ylabel="PC2")
axes[0,0].legend(fontsize=8)
eligible=[i for i,r in enumerate(sample_records) if r["history_len"]>=5 and len(set(r["labels"]))==2]
if not eligible:eligible=list(range(len(sample_records)))
example=int(np.random.default_rng(SEED).choice(eligible))
mask=latent["pair_group"]==example
palette=LinearSegmentedColormap.from_list("nrms_score_blues",plt.cm.Blues(np.linspace(.18,.95,256)))
dots=axes[0,1].scatter(news_xy[mask,0],news_xy[mask,1],c=latent["scores"][mask],cmap=palette,s=35)
axes[0,1].scatter(user_xy[example,0],user_xy[example,1],color=BLUE_DARK,marker="*",s=170,label="User")
clicked=mask&(latent["labels"]==1)
axes[0,1].scatter(news_xy[clicked,0],news_xy[clicked,1],facecolors="none",edgecolors=BLUE_DARK,s=100,label="Clicked news")
axes[0,1].set(title=f"One impression: actual dot-product score (sample {example})",xlabel="PC1",ylabel="PC2")
axes[0,1].legend(fontsize=8)
fig.colorbar(dots,ax=axes[0,1],label="Raw ranking score")

def draw_density(ax,values,title,xlabel):
    finite=np.isfinite(values)
    if not finite.any():
        ax.text(.5,.5,"Undefined: all vectors have zero norm",ha="center",transform=ax.transAxes)
        return np.nan
    bins=np.histogram_bin_edges(values[finite],bins=40)
    hist={}
    for label,color in ((0,BLUE_LIGHT),(1,BLUE_DARK)):
        subset=values[finite&(latent["labels"]==label)]
        if not len(subset):continue
        hist[label],_=np.histogram(subset,bins=bins,density=True)
        ax.hist(subset,bins=bins,density=True,color=color,alpha=.55,label="Clicked" if label else "Unclicked")
        ax.hist(subset,bins=bins,density=True,color=color,histtype="step")
    overlap=float(np.sum(np.minimum(hist[0],hist[1])*np.diff(bins))) if len(hist)==2 else np.nan
    ax.set(title=f"{title} (overlap ≈ {overlap:.2f})",xlabel=xlabel,ylabel="Density per label")
    ax.legend()
    return overlap

dot_overlap=draw_density(axes[1,0],latent["scores"],"Original scoring space","Dot product (actual model score)")
cos_overlap=draw_density(axes[1,1],cosine,"Directional similarity only","Cosine (not the scoring rule)")
fig.suptitle(f"{'Gated NRMS' if IS_GATED else 'NRMS'} | saved seed {SEED} | inference only")
savefig("nrms_user_news_latent.png")


## 6. Tables and provenance
Report score/cosine/norm distributions and sampled ranking metrics by history cohort. Sample metrics are diagnostics, not a replacement for the full-split evaluation in 7_1. Norms matter to dot product; high cosine alone does not imply a high score. In plain NRMS the zero-history vectors may all be zero; Gated NRMS combines the EMPTY-history representation with daily popularity.


In [ ]:
pair_history=np.array([r["history_len"] for r in sample_records])[latent["pair_group"]]
cohort=np.where(pair_history==0,"cold",np.where(pair_history<5,"sparse","warm"))
pair_table=pd.DataFrame(dict(record_index=sample_ids[latent["pair_group"]],
    impression_id=[sample_records[i]["impression_id"] for i in latent["pair_group"]],
    news_idx=latent["candidate_ids"],label=latent["labels"],cohort=cohort,
    score=latent["scores"],cosine=cosine,user_norm=user_norm,news_norm=news_norm,
    alpha=latent["alpha"][latent["pair_group"]]))
pair_table.to_csv(OUT_DIR/"nrms_pair_scores.csv",index=False)
distribution_rows=[]
for group,mask in [("all",np.ones(len(cohort),bool)),*[(g,cohort==g) for g in ("cold","sparse","warm")]]:
    for label in (0,1):
        sub=pair_table[mask&(pair_table.label.to_numpy()==label)]
        distribution_rows.append(dict(cohort=group,label="clicked" if label else "unclicked",n=len(sub),
            mean_score=sub.score.mean(),median_score=sub.score.median(),std_score=sub.score.std(),
            mean_cosine=sub.cosine.mean(),undefined_cosine=int(sub.cosine.isna().sum()),
            mean_user_norm=sub.user_norm.mean(),mean_news_norm=sub.news_norm.mean(),mean_alpha=sub.alpha.mean()))
distribution=pd.DataFrame(distribution_rows)
distribution.to_csv(OUT_DIR/"nrms_score_distribution.csv",index=False)
display(distribution)
ranking_rows=[]
for i,rec in enumerate(sample_records):
    mask=latent["pair_group"]==i
    auc,mrr,n5,n10=eval_impression(latent["labels"][mask],latent["scores"][mask])
    ranking_rows.append(dict(record_index=int(sample_ids[i]),history_len=rec["history_len"],
        auc=auc,mrr=mrr,ndcg5=n5,ndcg10=n10,alpha=float(latent["alpha"][i])))
ranking=pd.DataFrame(ranking_rows)
ranking.to_csv(OUT_DIR/"nrms_sample_ranking.csv",index=False)
display(ranking[["auc","mrr","ndcg5","ndcg10"]].mean().to_frame("sample_mean"))
def checkpoint_hash(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):h.update(chunk)
    return h.hexdigest()
(OUT_DIR/"nrms_latent_manifest.json").write_text(json.dumps(dict(checkpoint=str(CHECKPOINT),
    checkpoint_sha256=checkpoint_hash(CHECKPOINT),feature_id=feature_id,model="gated" if IS_GATED else "nrms",
    source_notebook="7_1_nrms_popularity_gated.ipynb",seed=SEED,split=SPLIT,
    sampled_impressions=len(sample_records),dimension=D_MODEL,heads=SOURCE_N_HEADS,
    dot_overlap=dot_overlap,cosine_overlap=cos_overlap,pca_retained_variance=float(pca.explained_variance_ratio_.sum()),
    verified_max_forward_error=latent["verified_max_error"],training_runs=0),indent=2))
print("Finished: checkpoint loaded and scored; zero training runs.")
print("Outputs:",OUT_DIR)
